<a href="https://colab.research.google.com/github/ravi-0309/Dynamic-Response/blob/main/Mode_Shapes_of_a_2D_Truss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
#########  SOLVING A 2-D TRUSS USING FEM  #########

import numpy as np
from scipy.linalg import eigh
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Geometric inputs
elements = int(input("Enter how many elements: ")) # Number of Elements
nodes = int(input("Enter the number of nodes: ")) # Number of Nodes
constraints = int(input("Enter the number of nodes that are constrained: "))
force = nodes - constraints
A = 100.0 # Area (mm2)
E = 100000.0 # Modulus of Elasticity (MPa)

# Initialize arrays
node_coordinates = np.zeros((nodes, 2))  # 2D coordinates (x, y)
element_connectivity = np.zeros((elements, 2), dtype=int)
constraints_x_y = np.zeros((constraints, 3))
element_orientation = np.zeros(elements)
element_length = np.zeros(elements)
element_stiffness_matrices = []
element_mass_matrices = []
element_direction = np.zeros((elements, 4))
global_stiffness_matrix = np.zeros((2*nodes, 2*nodes))
global_mass_matrix = np.zeros((2*nodes, 2*nodes))

# Loop to get coordinates for each node
for i in range(nodes):
  coord_input = input(f"Enter coordinates for node {i+1} (x y) in m: ").split()
  x = float(coord_input[0]) * 1000.0
  y = float(coord_input[1]) * 1000.0
  node_coordinates[i] = [x, y]

# Loop to get the node connectivity for each element
for i in range(elements):
  n1, n2 = map(int, input(f"Enter the node numbers for element {i+1} (start end): ").split())
  element_connectivity[i] = [n1, n2]

# Loop to get Boundary Conditions
for i in range(constraints):
  node, x, y = map(float, input(f"(n=node number, x,y=0 if no constraint, 1 if there is constraint (n x y)): ").split())
  constraints_x_y[i] = [node, x, y]

# Loop to get element directions
dof = np.arange(1, 2*nodes+1)
directions = dof.reshape(-1, 2)
for i in range(elements):
  n1, n2 = element_connectivity[i]
  a, b = directions[n1-1]
  c, d = directions[n2-1]
  element_direction[i] = [a-1, b-1, c-1, d-1]

########## END OF INPUTS #########

# Loop to calculate length, orientation and stiffness of each element
for i in range(elements):
  # Extracting Coordinates
  n1, n2 = element_connectivity[i]
  x1, y1 = node_coordinates[n1-1]
  x2, y2 = node_coordinates[n2-1]

  # Finding Length and Orientation
  L = round(np.sqrt((x2 - x1)**2 + (y2 - y1)**2), 2)
  theta = np.arctan2(y2 - y1, x2 - x1)
  element_length[i] = L
  element_orientation[i] = theta

  # Compute direction cosines
  c = round(np.cos(theta), 3)
  s = round(np.sin(theta), 3)

  # Compute element stiffness matrix
  k = (E * A / L) * np.array([
      [ c**2,  c*s, -c**2, -c*s],
      [ c*s,  s**2, -c*s, -s**2],
      [-c**2, -c*s,  c**2,  c*s],
      [-c*s, -s**2,  c*s,  s**2]
    ])
  sigma = 5 # kg/mm
  m = (sigma * L / 6.0) * np.array([
      [2, 0, 1, 0],
      [0, 2, 0, 1],
      [1, 0, 2, 0],
      [0, 1, 0, 2]
  ])

  # Store the stiffness matrix
  element_stiffness_matrices.append(k)
  element_mass_matrices.append(m)

# Generating Global Stiffness and Mass Matrix
for i in range(elements):
  indices = element_direction[i]
  for j in range(4):
    for k in range(4):
        global_stiffness_matrix[int(indices[j]), int(indices[k])] += round(element_stiffness_matrices[i][j, k], 3)
        global_mass_matrix[int(indices[j]), int(indices[k])] += round(element_mass_matrices[i][j, k], 3)

# Printing Element Stiffness Matrix of Each Element
print("Element Stiffness Matrices:")
for i, k in enumerate(element_stiffness_matrices):
  print(f"Stiffness Matrix of Element {i+1}:")
  print(k)

# Printing Element Mass Matrix of Each Element
print("Element Stiffness Matrices:")
for i, k in enumerate(element_mass_matrices):
  print(f"Mass Matrix of Element {i+1}:")
  print(k)

print("Global Stiffness Matrix:")
print(global_stiffness_matrix)
print("Global Mass Matrix:")
print(global_mass_matrix)

# Seggregating known and unknown dofs
constrained_dofs = []
free_dofs = []

# First identify all constrained DOFs
for i in range(constraints):
    node, x, y = constraints_x_y[i]
    node_idx = int(node) - 1  # convert to 0-based
    if x == 1: constrained_dofs.append(2*node_idx)
    if y == 1: constrained_dofs.append(2*node_idx + 1)

# Then free DOFs are all others
all_dofs = set(range(2*nodes))
free_dofs = sorted(list(all_dofs - set(constrained_dofs)))

# Extracting Sub Matrix for free DOFs of Stiffness and Mass
K = global_stiffness_matrix[free_dofs, :][:, free_dofs]
M = global_mass_matrix[free_dofs, :][:, free_dofs]

eigenvalues, phi = eigh(K, M)

wn = np.sqrt(eigenvalues)
fn = wn / (2* np.pi)

print("Natural Frequencies (Hz):")
print(wn)

print("Eigenvalues:")
print(eigenvalues)
print("Eigenvectors:")
print(phi)

len = __builtins__.len  # Restore len if overwritten

# MODE SHAPE VISUALIZATION AND ANIMATION
def animate_mode_shapes():
    num_modes = len(wn) if isinstance(wn, (np.ndarray, list)) else 1

    for mode in range(num_modes):
        actual_mode_number = mode + 1
        fig = plt.figure(figsize=(8, 6))
        plt.title(f"Mode {actual_mode_number} - Frequency: {fn[mode]:.3f} Hz", fontsize=12)

        # Initialize lines
        original_lines = []
        deformed_lines = []
        for elem in element_connectivity:
            orig_line, = plt.plot([], [], 'k--', alpha=0.5, linewidth=1)
            def_line, = plt.plot([], [], 'r-', linewidth=3)
            original_lines.append(orig_line)
            deformed_lines.append(def_line)

        # Reconstruct full mode shape
        full_mode = np.zeros(2*nodes)
        full_mode[free_dofs] = phi[:,mode]

        # Calculate scaling factor automatically
        max_disp = np.max(np.abs(full_mode))
        scale = 0.1 * np.max(node_coordinates) / (max_disp if max_disp > 0 else 1)

        # Set axis limits
        all_coords = node_coordinates.flatten()
        coord_range = max(all_coords.max() - all_coords.min(), 1)
        padding = 0.2 * coord_range

        plt.xlim(node_coordinates[:,0].min()-padding, node_coordinates[:,0].max()+padding)
        plt.ylim(node_coordinates[:,1].min()-padding, node_coordinates[:,1].max()+padding)

        plt.xlabel('X (mm)')
        plt.ylabel('Y (mm)')

        def init():
            # Initialize the animation
            for i, elem in enumerate(element_connectivity):
                n1, n2 = elem
                # Original structure
                x = [node_coordinates[n1-1,0], node_coordinates[n2-1,0]]
                y = [node_coordinates[n1-1,1], node_coordinates[n2-1,1]]
                original_lines[i].set_data(x, y)

                # Initial deformed position (same as original)
                deformed_lines[i].set_data(x, y)

            return original_lines + deformed_lines

        def update(frame):
            time = frame / 10
            displacement_factor = np.sin(2 * np.pi * time) * scale

            for i, elem in enumerate(element_connectivity):
                n1, n2 = elem

                # Original coordinates
                x1, y1 = node_coordinates[n1-1]
                x2, y2 = node_coordinates[n2-1]

                # Add scaled modal displacement
                dx1 = displacement_factor * full_mode[2*(n1-1)]
                dy1 = displacement_factor * full_mode[2*(n1-1)+1]

                dx2 = displacement_factor * full_mode[2*(n2-1)]
                dy2 = displacement_factor * full_mode[2*(n2-1)+1]

                # Update deformed lines
                deformed_lines[i].set_data(
                    [x1 + dx1, x2 + dx2],
                    [y1 + dy1, y2 + dy2]
                )

            return deformed_lines

        # Create animation
        ani = FuncAnimation(
            fig, update, frames=30, init_func=init,
            interval=50, blit=True, repeat=True
        )

        plt.close()
        print(f"\nMode {actual_mode_number} Animation - Scaling Factor: {scale:.2f}")
        display(HTML(ani.to_jshtml()))

# Execute the animation
print("\n=== Mode Shape Animations ===")
animate_mode_shapes()

Enter how many elements: 3
Enter the number of nodes: 3
Enter the number of nodes that are constrained: 2
Enter coordinates for node 1 (x y) in m: 0 0
Enter coordinates for node 2 (x y) in m: 0.5 0.866
Enter coordinates for node 3 (x y) in m: 1 0
Enter the node numbers for element 1 (start end): 1 2
Enter the node numbers for element 2 (start end): 3 2
Enter the node numbers for element 3 (start end): 1 3
(n=node number, x,y=0 if no constraint, 1 if there is constraint (n x y)): 1 1 1
(n=node number, x,y=0 if no constraint, 1 if there is constraint (n x y)): 3 0 1
Element Stiffness Matrices:
Stiffness Matrix of Element 1:
[[ 2500.050001    4330.08660173 -2500.050001   -4330.08660173]
 [ 4330.08660173  7499.7099942  -4330.08660173 -7499.7099942 ]
 [-2500.050001   -4330.08660173  2500.050001    4330.08660173]
 [-4330.08660173 -7499.7099942   4330.08660173  7499.7099942 ]]
Stiffness Matrix of Element 2:
[[ 2500.050001   -4330.08660173 -2500.050001    4330.08660173]
 [-4330.08660173  7499.


Mode 2 Animation - Scaling Factor: 7774.08



Mode 3 Animation - Scaling Factor: 7637.39


In [ ]:
13